# Análise · Reconciliação com os números do diretor
### V-Energy Grid 360 — primeiro passo do diagnóstico da receita

**Status: análise concluída.** Notebook de **análise**, fora do pipeline. Usa as Silvers do bloco B (`silver.leituras` e `silver.faturas`) para **reproduzir** o que o Henrique disse na diretoria e para localizar a queda de volume:

> "Comparando os últimos 12 meses com o primeiro ano da nossa base, o volume de energia faturada caiu quase 4% e a receita subiu só 8%."

**Resultado, em quatro linhas:**
1. As Silvers reproduzem os números dele: **receita +8,00%** e **volume −3,95%**.
2. A receita subiu porque o **preço médio por kWh subiu 12,45%** (tarifa + bandeira) e compensou o volume perdido: efeito-preço +R$ 14,09 mi, efeito-volume −R$ 4,66 mi.
3. A queda de volume vem das **3.510 UCs com painel solar** (geração distribuída): −61,2% (−11,2 milhões de kWh), enquanto as demais UCs cresceram +1,1%.
4. Dois meses (**novembro/2025 e fevereiro/2026**) têm consumo acima do ano anterior em todas as classes e escondem parte da queda.

| # | Seção | Pergunta |
|---|---|---|
| 1 | Preparação | As duas Silvers se juntam em uma linha por UC e mês? |
| 2 | Reconciliação | Os números batem com −4% e +8%? |
| 3 | Sensibilidade à troca de medidor | Quanto o tratamento das trocas muda o volume? |
| 4 | Preço × volume | Quanto da variação da receita é preço e quanto é volume? |
| 5 | Onde está a queda | Por classe (e município e bairro, quando houver a Silver de UCs) |
| 5.1 | Variação anual por classe e mês | Que classes caem, em que meses? |
| 5.2 | Meses atípicos | Há meses que fogem da tendência? |
| 6 | Painel de UCs estáveis | O volume cai mesmo nas UCs estáveis e sem troca de medidor? |
| 7 | Bandeira tarifária | O preço implícito muda com a bandeira? |
| 8 | Geração distribuída | As UCs com painel solar explicam a queda? |
| 9 | Resumo | O que levar para a daily e para o diagnóstico |

**Períodos** (por competência, `mes_referencia`): primeiro ano = `2022-07` a `2023-06` (P0); últimos 12 meses = `2025-07` a `2026-06` (P1).

**Limites desta análise:** as Silvers não têm as 32.285 faturas órfãs (1,5% da receita de energia). Com elas, a receita de energia variaria **+7,90%**, em vez de +8,00%. Quando a Silver de UCs ou de geração distribuída não existe no workspace, as seções 5 e 8 usam a Bronze (só a classe, e a lista de UCs com GD) e avisam. Tudo aqui mostra **associação**, e não causa.

## 1. Preparação

**O que faz:** lê as duas Silvers, confere as colunas e junta as duas por `(id_uc, mes_referencia)`. Como cada UC tem exatamente uma leitura e uma fatura por mês (garantido por duas checagens da Silver de faturas), o resultado deve ter **2.152.336 linhas**.

Os parâmetros no topo (nomes das tabelas e alvos do diretor) são o único lugar que talvez precise de ajuste.

**Achado:** as colunas esperadas existem (`silver.leituras` tem `consumo_kwh_original`, `troca_medidor` e `consumo_corrigido`; `silver.faturas` tem `em_aberto`). A junção tem **2.152.336 linhas**, como esperado. UC-meses: P0 536.267 e P1 539.768 (+0,65%), mais 1.076.301 fora dos dois períodos.

In [0]:
from pyspark.sql import functions as F
import pandas as pd

catalog = "v_energy"
silver_schema = f"{catalog}.silver"

# Tabelas das Silvers da Aimê (ajuste o nome se for diferente)
tabela_uc = f"{silver_schema}.unidades_consumidoras"
tabela_gd = f"{silver_schema}.geracao_distribuida"

# Períodos do diagnóstico (por competência) e números ditos pelo diretor
P0 = ("2022-07", "2023-06")   # primeiro ano
P1 = ("2025-07", "2026-06")   # últimos 12 meses
ALVO_VOLUME = -0.04           # "caiu quase 4%"
ALVO_RECEITA = 0.08           # "subiu só 8%"
TARIFA_RESIDENCIAL = 0.105    # reajuste homologado, só da tarifa residencial

def pct(x):
    return f"{x * 100:+.2f}%"
def n_br(n):
    return f"{n:,.0f}".replace(",", ".")

leit = spark.table(f"{silver_schema}.leituras")
fat = spark.table(f"{silver_schema}.faturas")

print("colunas de silver.leituras:", leit.columns)
print("colunas de silver.faturas: ", fat.columns)

exigidas = {
    "leituras": {"id_uc", "mes_referencia", "consumo_kwh", "consumo_kwh_original", "troca_medidor", "consumo_corrigido"},
    "faturas": {"id_uc", "mes_referencia", "valor_energia", "tributos", "valor_total", "bandeira_tarifaria"},
}
faltam = (exigidas["leituras"] - set(leit.columns)) | (exigidas["faturas"] - set(fat.columns))
assert not faltam, f"Colunas ausentes: {faltam}. Confira os nomes impressos acima."

colunas de silver.leituras: ['id_leitura', 'id_uc', 'id_medidor', 'mes_referencia', 'data_leitura', 'leitura_anterior', 'leitura_atual', 'consumo_kwh', 'consumo_kwh_original', 'tipo_leitura', 'leitura_anterior_ausente', 'consumo_corrigido', 'troca_medidor', 'source_file']
colunas de silver.faturas:  ['id_fatura', 'id_uc', 'mes_referencia', 'data_emissao', 'data_vencimento', 'data_pagamento', 'valor_energia', 'tributos', 'valor_total', 'bandeira_tarifaria', 'status', 'em_aberto', 'source_file']


In [0]:
# Uma linha por UC e mês: leitura + fatura
fato = (
    leit.select("id_uc", "mes_referencia", "consumo_kwh", "consumo_kwh_original", "troca_medidor", "consumo_corrigido")
    .join(fat.select("id_uc", "mes_referencia", "valor_energia", "tributos", "valor_total", "bandeira_tarifaria"),
          ["id_uc", "mes_referencia"], "inner")
    .withColumn("periodo",
                F.when(F.col("mes_referencia").between(*P0), "P0")
                 .when(F.col("mes_referencia").between(*P1), "P1"))
)

print("linhas da junção:", n_br(fato.count()), "(esperado: 2.152.336)")
print("UC-meses por período:")
fato.groupBy("periodo").count().orderBy("periodo").show()

linhas da junção: 2.152.336 (esperado: 2.152.336)
UC-meses por período:
+-------+-------+
|periodo|  count|
+-------+-------+
|   NULL|1076301|
|     P0| 536267|
|     P1| 539768|
+-------+-------+



## 2. Reconciliação com os números do diretor

**O que faz:** soma, em cada período, o volume (`consumo_kwh`) e a receita (`valor_energia` e `valor_total`) e calcula a variação P1 contra P0. Compara com −4% e +8%.

**Como ler:**
- **Volume** aparece em três versões, uma por tratamento das trocas de medidor: **A** (o que está na Silver: `consumo = leitura_atual`), **B** (consumo nulo nas trocas) e **C** (consumo original, sem tratar nada, com os valores negativos).
- Como a opção B tem menos linhas no P1 (as trocas ficam de fora), a tabela mostra também o **consumo médio por UC-mês**, que é a comparação justa.
- **Receita** não depende desses tratamentos, porque eles só mexem em consumo.
- `diferença` está em pontos percentuais (pp) contra o que o diretor disse. Perto de zero significa que reproduzimos o número.
- "Quase 4%" é um valor aproximado: diferenças de uns 0,5 pp não são motivo para desconfiar.

**Achado:** os números do diretor se reproduzem.

| Medida | P0 | P1 | Variação | Alvo | Diferença |
|---|---|---|---|---|---|
| Volume A (consumo tratado, Silver) | 227.052.927 kWh | 218.079.054 kWh | **−3,95%** | −4,00% | +0,05 pp |
| Volume B (consumo nulo nas trocas) | 227.052.927 | 215.436.332 | −5,12% | −4,00% | −1,12 pp |
| Volume C (consumo original, sem tratar) | 226.702.607 | −168.753.314 | −174,44% | −4,00% | −170,44 pp |
| Consumo médio por UC-mês, A | 423,40 | 404,02 | −4,58% | | |
| Consumo médio por UC-mês, B | 423,40 | 403,84 | −4,62% | | |
| Consumo médio por UC-mês, C | 422,74 | −312,64 | −173,96% | | |
| Receita de energia (`valor_energia`) | R$ 117,83 mi | R$ 127,26 mi | **+8,00%** | +8,00% | 0,00 |
| Receita total (`valor_total`) | R$ 157,11 mi | R$ 169,68 mi | **+8,00%** | +8,00% | 0,00 |

`valor_energia` e `valor_total` dão a mesma variação, então a escolha entre as duas definições de receita não muda o diagnóstico.

In [0]:
agg = (
    fato.filter("periodo IS NOT NULL")
    .groupBy("periodo")
    .agg(
        F.countDistinct("id_uc").alias("ucs"),
        F.count("*").alias("uc_meses"),
        F.sum(F.when(~F.col("troca_medidor"), 1).otherwise(0)).alias("uc_meses_sem_troca"),
        F.sum("consumo_kwh").cast("double").alias("vol_A"),
        F.sum(F.when(~F.col("troca_medidor"), F.col("consumo_kwh"))).cast("double").alias("vol_B"),
        F.sum("consumo_kwh_original").cast("double").alias("vol_C"),
        F.sum(F.col("valor_energia").cast("double")).alias("rec_energia"),
        F.sum(F.col("valor_total").cast("double")).alias("rec_total"),
    )
    .toPandas().set_index("periodo")
)

def var(coluna):
    return agg.loc["P1", coluna] / agg.loc["P0", coluna] - 1

media = lambda per, vol, linhas: agg.loc[per, vol] / agg.loc[per, linhas]

linhas = [
    ("Volume A: consumo tratado (Silver)",             agg.loc["P0", "vol_A"], agg.loc["P1", "vol_A"], ALVO_VOLUME),
    ("Volume B: consumo nulo nas trocas",              agg.loc["P0", "vol_B"], agg.loc["P1", "vol_B"], ALVO_VOLUME),
    ("Volume C: consumo original, sem tratar",         agg.loc["P0", "vol_C"], agg.loc["P1", "vol_C"], ALVO_VOLUME),
    ("Consumo médio por UC-mês, A",                    media("P0", "vol_A", "uc_meses"), media("P1", "vol_A", "uc_meses"), None),
    ("Consumo médio por UC-mês, B (sem os meses de troca)", media("P0", "vol_B", "uc_meses_sem_troca"), media("P1", "vol_B", "uc_meses_sem_troca"), None),
    ("Consumo médio por UC-mês, C",                    media("P0", "vol_C", "uc_meses"), media("P1", "vol_C", "uc_meses"), None),
    ("Receita de energia (valor_energia)",             agg.loc["P0", "rec_energia"], agg.loc["P1", "rec_energia"], ALVO_RECEITA),
    ("Receita total (valor_total)",                    agg.loc["P0", "rec_total"], agg.loc["P1", "rec_total"], ALVO_RECEITA),
]
tabela = pd.DataFrame([
    {"medida": nome, "P0": v0, "P1": v1, "variação": v1 / v0 - 1,
     "alvo do diretor": alvo, "diferença (pp)": (None if alvo is None else (v1 / v0 - 1 - alvo) * 100)}
    for nome, v0, v1, alvo in linhas
])
fmt = tabela.copy()
for c in ["P0", "P1"]:
    fmt[c] = fmt[c].map(lambda x: f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))
fmt["variação"] = tabela["variação"].map(pct)
fmt["alvo do diretor"] = tabela["alvo do diretor"].map(lambda x: "" if pd.isna(x) else pct(x))
fmt["diferença (pp)"] = tabela["diferença (pp)"].map(lambda x: "" if pd.isna(x) else f"{x:+.2f}")
display(fmt)

print("UCs com fatura em P0:", n_br(agg.loc["P0", "ucs"]), "| em P1:", n_br(agg.loc["P1", "ucs"]))

medida,P0,P1,variação,alvo do diretor,diferença (pp)
Volume A: consumo tratado (Silver),"227.052.927,00","218.079.054,00",-3.95%,-4.00%,+0.05
Volume B: consumo nulo nas trocas,"227.052.927,00","215.436.332,00",-5.12%,-4.00%,-1.12
"Volume C: consumo original, sem tratar","226.702.607,00","-168.753.314,00",-174.44%,-4.00%,-170.44
"Consumo médio por UC-mês, A","423,40","404,02",-4.58%,,
"Consumo médio por UC-mês, B (sem os meses de troca)","423,40","403,84",-4.62%,,
"Consumo médio por UC-mês, C","422,74","-312,64",-173.96%,,
Receita de energia (valor_energia),"117.830.394,38","127.262.595,65",+8.00%,+8.00%,+0.00
Receita total (valor_total),"157.107.191,43","169.683.458,59",+8.00%,+8.00%,+0.00


UCs com fatura em P0: 44.741 | em P1: 45.000


## 3. Sensibilidade à troca de medidor

**O que faz:** isola o efeito do tratamento das trocas sobre a variação de volume, usando as três versões da seção 2. Se as três variações forem parecidas, a escolha do tratamento é **irrelevante para o diagnóstico**. Se forem muito diferentes, o grupo precisa saber que o número de volume depende dela.

**Como ler:**
- `A − B` e `A − C` em pontos percentuais: quanto a escolha da `leitura_atual` desloca a variação de volume em relação às alternativas.
- A opção C (consumo original) tem valores negativos grandes (até −190 mil kWh por linha), então tende a distorcer o P1 para baixo. Se C ficar perto do −4% do diretor, é um sinal de que a fonte do número dele pode não ter tratado essas linhas. Essa é uma hipótese a investigar, e não uma conclusão.

**Achado:** são 6.299 UC-meses de troca de medidor (1,17% do P1).
- Variação do volume total: A −3,95%, B −5,12%, C −174,44%. A − B = +1,16 pp, e A − C = +170,49 pp.
- Variação do **consumo médio por UC-mês** (a comparação justa, porque B tem menos linhas): A −4,58% e B −4,62%. A escolha entre `leitura_atual` e consumo nulo praticamente não muda o diagnóstico por UC.
- **A é a única que fecha o "quase −4%"** do diretor no total. O consumo original (C) distorce tudo, por causa dos valores negativos da troca de medidor.

In [0]:
vA, vB, vC = var("vol_A"), var("vol_B"), var("vol_C")
mA = media("P1", "vol_A", "uc_meses") / media("P0", "vol_A", "uc_meses") - 1
mB = media("P1", "vol_B", "uc_meses_sem_troca") / media("P0", "vol_B", "uc_meses_sem_troca") - 1

print(f"Variação do volume total:     A {pct(vA)} | B {pct(vB)} | C {pct(vC)}")
print(f"Variação do consumo médio/UC-mês: A {pct(mA)} | B {pct(mB)}")
print(f"A − B: {(vA - vB) * 100:+.2f} pp   |   A − C: {(vA - vC) * 100:+.2f} pp")
print("UC-meses de troca de medidor:", n_br(agg.loc["P1", "uc_meses"] - agg.loc["P1", "uc_meses_sem_troca"]),
      f"({(1 - agg.loc['P1', 'uc_meses_sem_troca'] / agg.loc['P1', 'uc_meses']) * 100:.2f}% do P1)")

Variação do volume total:     A -3.95% | B -5.12% | C -174.44%
Variação do consumo médio/UC-mês: A -4.58% | B -4.62%
A − B: +1.16 pp   |   A − C: +170.49 pp
UC-meses de troca de medidor: 6.299 (1.17% do P1)


## 4. Preço × volume

**O que faz:** decompõe a variação da receita de energia em efeito-volume e efeito-preço, com as fórmulas do contrato da Gold:
- preço médio `P = receita / volume`;
- `efeito_volume = (V1 − V0) × P0`;
- `efeito_preco = (P1 − P0) × V1`;
- os dois somam exatamente `receita_1 − receita_0`.

Usa o volume **A** (o da Silver) e `valor_energia` (sem tributos, que é a sugestão do contrato). Também compara a variação do preço médio com os 10,5% da tarifa residencial.

**Como ler:** se o preço médio sobe mais do que 10,5%, a diferença não vem do reajuste da tarifa residencial. Pode ser bandeira tarifária (seção 7) ou mudança no mix de classes (seção 5, que decompõe por classe). Lembre que os 10,5% valem **só para a classe residencial**.

**Achado:** o preço médio subiu de R$ 0,5190 para **R$ 0,5836 por kWh (+12,45%)**, contra os 10,5% de tarifa residencial.
- Variação da receita de energia: **R$ 9.432.201,27**.
- **Efeito-volume: −R$ 4.657.041,90** (−49,4% da variação). **Efeito-preço: +R$ 14.089.243,17** (+149,4%). A conferência (efeito-volume + efeito-preço − variação) dá zero.

In [0]:
V0, V1 = agg.loc["P0", "vol_A"], agg.loc["P1", "vol_A"]
R0, R1 = agg.loc["P0", "rec_energia"], agg.loc["P1", "rec_energia"]
Pm0, Pm1 = R0 / V0, R1 / V1

ef_volume = (V1 - V0) * Pm0
ef_preco = (Pm1 - Pm0) * V1
var_receita = R1 - R0

print(f"Preço médio P0: R$ {Pm0:.4f}/kWh | P1: R$ {Pm1:.4f}/kWh | variação {pct(Pm1 / Pm0 - 1)}  (tarifa residencial: {pct(TARIFA_RESIDENCIAL)})")
print(f"Variação da receita de energia: R$ {var_receita:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))
print(f"  efeito volume: R$ {ef_volume:,.2f} ({ef_volume / var_receita * 100:.1f}% da variação)".replace(",", "X").replace(".", ",").replace("X", "."))
print(f"  efeito preço:  R$ {ef_preco:,.2f} ({ef_preco / var_receita * 100:.1f}% da variação)".replace(",", "X").replace(".", ",").replace("X", "."))
print(f"  conferência (efeito volume + efeito preço − variação): {ef_volume + ef_preco - var_receita:.6f}  (esperado: 0)")

Preço médio P0: R$ 0.5190/kWh | P1: R$ 0.5836/kWh | variação +12.45%  (tarifa residencial: +10.50%)
Variação da receita de energia: R$ 9.432.201,27
  efeito volume: R$ -4.657.041,90 (-49,4% da variação)
  efeito preço:  R$ 14.089.243,17 (149,4% da variação)
  conferência (efeito volume + efeito preço − variação): 0.000000  (esperado: 0)


## 5. Onde está a queda

**O que faz:** repete a decomposição por classe, por município e por bairro (município + bairro, porque o mesmo nome de bairro existe em municípios diferentes). Precisa da Silver de UCs, com as colunas `classe`, `municipio` e `bairro`.

**Como ler:**
- Ordenado por `efeito_volume` do menor para o maior: as primeiras linhas são onde mais energia deixou de ser vendida.
- `ucs_P0` e `ucs_P1` mostram se a base cresceu naquele grupo. Um grupo que ganha UCs e perde volume tem consumo médio por UC em queda forte.
- **Efeito de composição (mix):** a soma dos efeitos por classe não é igual ao efeito agregado, porque a mistura entre classes mudou. A diferença é impressa no fim e responde "parte da variação é só mudança de mix".

**Conferência embutida:** quantas linhas ficam sem classe ao juntar com a Silver de UCs. O esperado é **0**; se for maior, há UCs das leituras e faturas que não existem na Silver de UCs (as 200 UCs de teste que saíram dela, por exemplo).

**Achado (P1 contra P0, volume A, receita = `valor_energia`):**

| Classe | UCs P0 → P1 | Volume | Preço por kWh | Receita | Efeito-volume |
|---|---|---|---|---|---|
| Comercial | 4.882 → 4.909 | **−10,15%** | +18,79% | +6,72% | **−R$ 2,98 mi** |
| Residencial | 38.114 → 38.336 | **−5,36%** | +13,01% | +6,95% | **−R$ 2,35 mi** |
| Industrial | 875 → 877 | +0,59% | +8,01% | +8,65% | +R$ 0,24 mi |
| Rural | 649 → 656 | +2,52% | +25,97% | +29,14% | +R$ 0,03 mi |
| Poder público | 221 → 222 | +2% a +3% | +19% | +22% | +R$ 0,06 mi |

- A perda de volume está em duas classes: comercial e residencial somam −R$ 5,32 mi, e as outras três compensam +R$ 0,33 mi.
- O preço residencial subiu 13,01%, contra 10,5% de tarifa: a diferença (~2,3%) é compatível com a bandeira (estimativa).
- Os reajustes das outras classes não são conhecidos (o case só cita o residencial).
- **Efeito de composição (mix):** a soma dos efeitos-volume por classe é −R$ 4.994.044,40, contra −R$ 4.657.041,90 do agregado. A diferença, +R$ 337.002,50, mostra que os kWh perdidos estão em classes com preço por kWh acima da média.
- As UCs somam 44.741 no P0 e **45.000 no P1** (o número do case). As 200 UCs de teste do cadastro não aparecem em leituras nem em faturas.
- Os números de Poder público vêm de uma execução em que a Bronze tinha duas grafias (`PODER PUBLICO` e `PODER PÚBLICO`), agora unificadas no notebook.
- Município e bairro dependem da Silver de UCs (ainda não rodada neste workspace).

In [0]:
tem_uc = False
try:
    uc = spark.table(tabela_uc)
    assert "classe" in uc.columns, f"{tabela_uc} não tem a coluna 'classe'. Colunas: {uc.columns}"
    geo = [c for c in ["classe", "municipio", "bairro"] if c in uc.columns]
    uc = uc.select("id_uc", *geo).dropDuplicates(["id_uc"])
    origem_uc = "Silver de UCs"
except Exception as e:
    print(f"Silver de UCs indisponível ({str(e)[:90]}...). Usando a Bronze, só com a classe.")
    uc = (spark.table(f"{catalog}.bronze.unidades_consumidoras")
            .filter(F.col("id_uc") != "id_uc")   # a Bronze tem uma linha de cabeçalho repetido
            .select("id_uc", F.regexp_replace(F.upper(F.trim("classe")), "Ú", "U").alias("classe"))  # PODER PÚBLICO / PUBLICO
            .dropDuplicates(["id_uc"]))
    geo = ["classe"]
    origem_uc = "Bronze (só a classe; município e bairro não são confiáveis lá)"

fato_uc = fato.join(uc, "id_uc", "left")
sem_classe = fato_uc.filter(F.col("classe").isNull()).select("id_uc").distinct().count()
print(f"origem das UCs: {origem_uc}")
print(f"UCs de leituras/faturas sem correspondência no cadastro: {sem_classe} (esperado: 0)")
uc.groupBy("classe").count().show()   # confira se as classes aparecem sem variações de grafia
tem_uc = True

origem das UCs: Silver de UCs
UCs de leituras/faturas sem correspondência no cadastro: 0 (esperado: 0)
+-------------+-----+
|       classe|count|
+-------------+-----+
|    COMERCIAL| 4909|
|  RESIDENCIAL|38336|
|PODER PUBLICO|  222|
|        RURAL|  656|
|   INDUSTRIAL|  877|
+-------------+-----+



In [0]:
def decompor(df, chaves):
    """Decomposição preço × volume por grupo, P1 contra P0 (volume A, receita = valor_energia)."""
    b = (df.filter("periodo IS NOT NULL").groupBy(*chaves, "periodo")
           .agg(F.sum("consumo_kwh").cast("double").alias("v"),
                F.sum(F.col("valor_energia").cast("double")).alias("r"),
                F.countDistinct("id_uc").alias("ucs")))
    p = (b.groupBy(*chaves).pivot("periodo", ["P0", "P1"])
          .agg(F.first("v").alias("v"), F.first("r").alias("r"), F.first("ucs").alias("ucs")))
    return (p.withColumn("pm0", F.col("P0_r") / F.col("P0_v"))
             .withColumn("pm1", F.col("P1_r") / F.col("P1_v"))
             .withColumn("var_volume_pct", F.round((F.col("P1_v") / F.col("P0_v") - 1) * 100, 2))
             .withColumn("var_receita_pct", F.round((F.col("P1_r") / F.col("P0_r") - 1) * 100, 2))
             .withColumn("var_preco_pct", F.round((F.col("pm1") / F.col("pm0") - 1) * 100, 2))
             .withColumn("efeito_volume", F.round((F.col("P1_v") - F.col("P0_v")) * F.col("pm0"), 2))
             .withColumn("efeito_preco", F.round((F.col("pm1") - F.col("pm0")) * F.col("P1_v"), 2))
             .withColumn("ucs_P0", F.col("P0_ucs")).withColumn("ucs_P1", F.col("P1_ucs"))
             .select(*chaves, "ucs_P0", "ucs_P1", "var_volume_pct", "var_receita_pct", "var_preco_pct",
                     "efeito_volume", "efeito_preco"))

if tem_uc:
    print("Por classe:")
    por_classe = decompor(fato_uc, ["classe"])
    display(por_classe.orderBy("efeito_volume"))

    soma_vol_classes = por_classe.agg(F.sum("efeito_volume")).first()[0]
    print(f"Soma dos efeitos-volume por classe: R$ {soma_vol_classes:,.2f} | efeito-volume agregado (seção 4): R$ {ef_volume:,.2f}"
          .replace(",", "X").replace(".", ",").replace("X", "."))
    print(f"Efeito de composição (mix) = agregado − soma por classe: R$ {ef_volume - soma_vol_classes:,.2f}"
          .replace(",", "X").replace(".", ",").replace("X", "."))

Por classe:


classe,ucs_P0,ucs_P1,var_volume_pct,var_receita_pct,var_preco_pct,efeito_volume,efeito_preco
COMERCIAL,4882,4909,-10.15,6.72,18.79,-2976246.78,4947303.95
RESIDENCIAL,38114,38336,-5.36,6.95,13.01,-2346858.38,5387521.92
RURAL,649,656,2.52,29.14,25.97,30174.89,318988.62
PODER PUBLICO,221,222,2.43,21.74,18.86,56426.73,449215.09
INDUSTRIAL,875,877,0.59,8.65,8.01,242458.88,3323216.35


Soma dos efeitos-volume por classe: R$ -4.994.044,66 | efeito-volume agregado (seção 4): R$ -4.657.041,90
Efeito de composição (mix) = agregado − soma por classe: R$ 337.002,76


In [0]:
if tem_uc and "municipio" in geo:
    print("Por município:")
    display(decompor(fato_uc, ["municipio"]).orderBy("efeito_volume"))
if tem_uc and {"municipio", "bairro"} <= set(geo):
    print("15 piores bairros (município + bairro) por efeito-volume:")
    display(decompor(fato_uc, ["municipio", "bairro"]).orderBy("efeito_volume").limit(15))

Por município:


municipio,ucs_P0,ucs_P1,var_volume_pct,var_receita_pct,var_preco_pct,efeito_volume,efeito_preco
Recife,17903,17995,-4.35,7.68,12.57,-2040492.49,5643415.34
Jaboatão dos Guararapes,7755,7800,-6.75,4.75,12.34,-1343605.04,2288611.46
Paulista,3998,4024,-2.64,9.54,12.51,-267602.01,1234888.75
Cabo de Santo Agostinho,2491,2513,-3.22,8.85,12.47,-228124.88,855133.09
Igarassu,1393,1400,-4.09,8.35,12.97,-135078.0,411063.17
Olinda,4192,4212,-1.19,10.71,12.04,-133722.7,1342197.57
Camaragibe,1773,1783,-2.38,9.73,12.41,-118217.48,600529.92
Ipojuca,1167,1176,-2.71,9.84,12.89,-81777.69,378879.77
São Lourenço da Mata,1388,1398,-1.98,10.1,12.32,-74941.43,457399.72
Moreno,668,672,-3.21,7.98,11.56,-67689.14,235926.15


15 piores bairros (município + bairro) por efeito-volume:


municipio,bairro,ucs_P0,ucs_P1,var_volume_pct,var_receita_pct,var_preco_pct,efeito_volume,efeito_preco
Recife,Iputinga,1224,1234,-18.4,-10.69,9.45,-660479.93,276825.72
Jaboatão dos Guararapes,Jardim Jordão,1103,1111,-21.58,-13.24,10.64,-558201.68,215842.33
Jaboatão dos Guararapes,Prazeres,665,668,-21.78,-12.41,11.97,-343489.59,147722.15
Recife,Macaxeira,558,561,-16.77,-8.28,10.2,-272071.08,137728.6
Recife,Brasília Teimosa,2125,2130,-3.08,9.29,12.77,-171996.84,691257.69
Jaboatão dos Guararapes,Barra de Jangada,685,689,-4.92,7.34,12.9,-87901.75,218942.92
Jaboatão dos Guararapes,Engenho Velho,518,519,-7.34,5.3,13.64,-81006.68,139518.25
Jaboatão dos Guararapes,Sucupira,630,632,-3.6,7.91,11.94,-71082.57,227166.35
Cabo de Santo Agostinho,Área Rural do Cabo de Santo Agostinho,265,267,-10.57,0.92,12.85,-67327.83,73202.17
Camaragibe,Vila da Fábrica,471,476,-5.54,7.0,13.27,-62279.79,140971.91


## 5.1 Variação anual por classe e mês

**O que faz:** para cada mês de P1, compara o consumo de cada classe com o **mesmo mês do ano anterior** (o consumo é sazonal: pico em janeiro e fevereiro, vale em julho e agosto, então não se compara um mês com o vizinho).

**Achado:** variação do consumo contra o mesmo mês do ano anterior (%):

| Mês | Comercial | Industrial | Poder público | Residencial | Rural |
|---|---|---|---|---|---|
| 2025-07 | −7,9 | 0,0 | −1,5 | −5,3 | +0,4 |
| 2025-08 | −8,1 | −0,2 | 0,0 | −5,3 | +0,3 |
| 2025-09 | −7,7 | +0,2 | −0,7 | −5,2 | −0,4 |
| 2025-10 | −7,1 | 0,0 | 0,0 | −5,1 | −0,7 |
| **2025-11** | **+5,5** | **+4,1** | **+11,5** | **+14,4** | **+9,1** |
| 2025-12 | −7,0 | +0,4 | −0,9 | −4,7 | −0,3 |
| 2026-01 | −7,0 | −0,4 | −0,3 | −4,7 | −0,4 |
| **2026-02** | **+5,4** | **+3,4** | **+10,7** | **+13,7** | **+8,8** |
| 2026-03 | −6,6 | +0,3 | −0,5 | −4,4 | +0,6 |
| 2026-04 | −6,4 | +0,5 | +0,8 | −4,3 | +0,4 |
| 2026-05 | −5,0 | −0,3 | 0,0 | −3,5 | −0,3 |
| 2026-06 | −2,8 | +0,3 | +2,1 | −2,6 | −0,1 |

- Nos meses normais, **o residencial cai ~5% e o comercial ~7% ao ano**. Industrial, rural e poder público ficam parados.
- **Novembro/2025 e fevereiro/2026 viram o jogo em todas as classes ao mesmo tempo**, com mais força no residencial (de ~−4,7% para +14%).
- A queda parece diminuir de abril a junho de 2026 (residencial de −4,3% para −2,6%). Pode ser estabilização ou efeito da base de comparação.

In [0]:
meses_p1 = [f"{a}-{m:02d}" for a, m in [(2025,7),(2025,8),(2025,9),(2025,10),(2025,11),(2025,12),(2026,1),(2026,2),(2026,3),(2026,4),(2026,5),(2026,6)]]
meses_ant = [f"{int(m[:4]) - 1}-{m[5:]}" for m in meses_p1]

base_cm = (spark.table(f"{silver_schema}.leituras").join(uc, "id_uc", "left")
             .filter(F.col("mes_referencia").isin(meses_p1 + meses_ant))
             .groupBy("classe", "mes_referencia").agg(F.sum("consumo_kwh").alias("kwh")))

atual = base_cm.filter(F.col("mes_referencia").isin(meses_p1)).withColumnRenamed("kwh", "kwh_atual")
ant = (base_cm.filter(F.col("mes_referencia").isin(meses_ant))
           .withColumn("mes_referencia", F.concat((F.substring("mes_referencia", 1, 4).cast("int") + 1).cast("string"), F.substring("mes_referencia", 5, 3)))
           .withColumnRenamed("kwh", "kwh_ant"))

(atual.join(ant, ["classe", "mes_referencia"])
      .withColumn("yoy_pct", F.round((F.col("kwh_atual") / F.col("kwh_ant") - 1) * 100, 1))
      .groupBy("mes_referencia").pivot("classe").agg(F.first("yoy_pct"))
      .orderBy("mes_referencia").show(12))

+--------------+---------+----------+-------------+-----------+-----+
|mes_referencia|COMERCIAL|INDUSTRIAL|PODER PUBLICO|RESIDENCIAL|RURAL|
+--------------+---------+----------+-------------+-----------+-----+
|       2025-07|     -7.9|       0.0|         -1.5|       -5.3|  0.4|
|       2025-08|     -8.1|      -0.2|          0.0|       -5.3|  0.3|
|       2025-09|     -7.7|       0.2|         -0.7|       -5.2| -0.4|
|       2025-10|     -7.1|       0.0|          0.0|       -5.1| -0.7|
|       2025-11|      5.5|       4.1|         11.5|       14.4|  9.1|
|       2025-12|     -7.0|       0.4|         -0.9|       -4.7| -0.3|
|       2026-01|     -7.0|      -0.4|         -0.3|       -4.7| -0.4|
|       2026-02|      5.4|       3.4|         10.7|       13.7|  8.8|
|       2026-03|     -6.6|       0.3|         -0.5|       -4.4|  0.6|
|       2026-04|     -6.4|       0.5|          0.8|       -4.3|  0.4|
|       2026-05|     -5.0|      -0.3|          0.0|       -3.5| -0.3|
|       2026-06|    

## 5.2 Meses atípicos: novembro de 2025 e fevereiro de 2026

**O que faz:** compara o consumo **total** de cada mês de P1 com o mesmo mês do ano anterior e estima o que seria o P1 se os dois meses atípicos seguissem a tendência dos outros dez. Também mostra os totais anuais.

**Achado:**
- Os outros dez meses de P1 ficam de **−1,2% a −3,6%** (média −2,94%) contra o ano anterior. **Novembro/2025 (+8,5%) e fevereiro/2026 (+8,0%)** destoam, e são os mesmos meses em que as ocorrências de rede também têm um pico menor (~5.300, 1,6× o normal, em todas as causas).
- Consumo por ano (julho a junho): 227,05 → 226,37 → 219,95 → 218,08 milhões de kWh (−0,3%, −2,8% e −0,85%). A última variação está **atenuada** pelos dois meses atípicos.
- **Hipótese:** se os dois meses seguissem a tendência dos demais (−2,94%), o P1 seria de ~213,4 milhões de kWh, e a queda sobre o primeiro ano seria de **~−6,0%, e não −3,95%**. Isso supõe que os dois meses "deveriam" ter caído como os outros. A causa dos picos (calor? outra?) não está nos dados.

In [0]:
mensal = (spark.table(f"{silver_schema}.leituras").groupBy("mes_referencia")
          .agg(F.sum("consumo_kwh").cast("double").alias("kwh")).orderBy("mes_referencia")
          .toPandas().set_index("mes_referencia")["kwh"])

def mes_ant(m):
    return f"{int(m[:4]) - 1}-{m[5:]}"

yoy = pd.Series({m: mensal[m] / mensal[mes_ant(m)] - 1 for m in meses_p1})
print("variação contra o mesmo mês do ano anterior (%):")
print((yoy * 100).round(1).to_string())

ATIPICOS = ["2025-11", "2026-02"]
tendencia = yoy.drop(ATIPICOS).mean()
p0_total = mensal[(mensal.index >= P0[0]) & (mensal.index <= P0[1])].sum()
p1_total = mensal[(mensal.index >= P1[0]) & (mensal.index <= P1[1])].sum()
p1_ajustado = p1_total - sum(mensal[m] for m in ATIPICOS) + sum(mensal[mes_ant(m)] * (1 + tendencia) for m in ATIPICOS)

print(f"\ntendência dos outros 10 meses: {pct(tendencia)}")
print(f"P1 observado: {p1_total / 1e6:.2f} mi kWh ({pct(p1_total / p0_total - 1)} contra P0)")
print(f"P1 se nov/25 e fev/26 seguissem a tendência: {p1_ajustado / 1e6:.2f} mi kWh ({pct(p1_ajustado / p0_total - 1)} contra P0)")

anos = [mensal.iloc[i * 12:(i + 1) * 12].sum() for i in range(4)]
print("\nconsumo por ano (jul a jun), em milhões de kWh:", [round(a / 1e6, 2) for a in anos])
print("variação entre anos:", [pct(anos[i + 1] / anos[i] - 1) for i in range(3)])

variação contra o mesmo mês do ano anterior (%):
2025-07   -3.2
2025-08   -3.2
2025-09   -3.1
2025-10   -3.4
2025-11    8.5
2025-12   -3.3
2026-01   -3.6
2026-02    8.0
2026-03   -3.1
2026-04   -2.9
2026-05   -2.4
2026-06   -1.2

tendência dos outros 10 meses: -2.94%
P1 observado: 218.08 mi kWh (-3.95% contra P0)
P1 se nov/25 e fev/26 seguissem a tendência: 213.42 mi kWh (-6.00% contra P0)

consumo por ano (jul a jun), em milhões de kWh: [np.float64(227.05), np.float64(226.37), np.float64(219.95), np.float64(218.08)]
variação entre anos: ['-0.30%', '-2.83%', '-0.85%']


## 6. Painel de UCs estáveis

**O que faz:** isola um grupo "limpo" para testar se o volume cai **por comportamento de consumo**, e não por mudança na base ou por tratamento de dado. Usa só as UCs que têm os 12 meses em P0 **e** os 12 meses em P1 e separa as que tiveram troca de medidor das que não tiveram.

**Como ler:**
- Compare `consumo médio por UC-mês` entre P0 e P1 nas **UCs sem troca**: nelas, nenhum tratamento de dado interfere. Se o consumo médio cair, a queda é real.
- Nas UCs com troca, o P1 inclui os meses tratados (`consumo = leitura_atual`). Uma diferença grande entre os dois grupos indica que o tratamento pesa.
- O número de UCs do painel mostra o quanto da base ele representa.

**Achado:** 38.397 UCs têm os 12 meses nos dois períodos e **nenhuma troca de medidor**; 6.241 UCs têm os 12 meses e tiveram troca.
- UCs **sem** troca: consumo médio por UC-mês 427,55 → 407,98 kWh (**−4,58%**). Nelas nenhum tratamento de dado interfere, então a queda é de comportamento de consumo.
- UCs **com** troca: 398,49 → 384,79 (−3,44%).
- A base não encolheu: as UCs com fatura foram de 44.741 (P0) para 45.000 (P1). Cada UC consome menos, e não há menos UCs.

In [0]:
meses = (fato.filter("periodo IS NOT NULL").groupBy("id_uc", "periodo").agg(F.count("*").alias("meses")))
completas = (meses.groupBy("id_uc").pivot("periodo", ["P0", "P1"]).agg(F.first("meses"))
                  .filter("P0 = 12 AND P1 = 12").select("id_uc"))
com_troca = fato.filter("troca_medidor").select("id_uc").distinct().withColumn("teve_troca", F.lit(True))

painel = (fato.filter("periodo IS NOT NULL").join(completas, "id_uc", "left_semi")
              .join(com_troca, "id_uc", "left")
              .withColumn("teve_troca", F.coalesce("teve_troca", F.lit(False))))

p = (painel.groupBy("teve_troca", "periodo")
           .agg(F.countDistinct("id_uc").alias("ucs"),
                F.sum("consumo_kwh").cast("double").alias("volume"),
                F.count("*").alias("uc_meses"))
           .withColumn("consumo_medio_uc_mes", F.round(F.col("volume") / F.col("uc_meses"), 2))
           .orderBy("teve_troca", "periodo"))
display(p)

pd_p = p.toPandas().set_index(["teve_troca", "periodo"])
for grupo in [False, True]:
    try:
        a, b = pd_p.loc[(grupo, "P0"), "consumo_medio_uc_mes"], pd_p.loc[(grupo, "P1"), "consumo_medio_uc_mes"]
        print(f"UCs {'COM' if grupo else 'SEM'} troca: consumo médio por UC-mês {a:.2f} → {b:.2f} ({pct(b / a - 1)})")
    except KeyError:
        print(f"Sem dados para o grupo teve_troca={grupo}")

teve_troca,periodo,ucs,volume,uc_meses,consumo_medio_uc_mes
false,P0,38397,1.96997493E8,460764,427.55
false,P1,38397,1.87982453E8,460764,407.98
true,P0,6241,2.9844051E7,74892,398.49
true,P1,6241,2.8817587E7,74892,384.79


UCs SEM troca: consumo médio por UC-mês 427.55 → 407.98 (-4.58%)
UCs COM troca: consumo médio por UC-mês 398.49 → 384.79 (-3.44%)


## 7. Bandeira tarifária

**O que faz:** mostra, em cada período, quantas faturas há por bandeira e o preço implícito mediano (`valor_energia / consumo_kwh`). A `silver.faturas` tem a bandeira só como rótulo, sem uma coluna com o valor do adicional, então o efeito dela está embutido em `valor_energia` e só dá para **estimar** pelo preço por kWh.

**Como ler:**
- A distribuição de faturas por bandeira mostra se P1 teve mais bandeira cara que P0.
- Se o preço mediano da mesma bandeira sobe de P0 para P1, isso é reajuste de tarifa. Se o preço sobe porque o mix de bandeiras mudou, é efeito de bandeira. Os dois entram no efeito-preço da seção 4.
- Meses de troca de medidor e consumo não positivo ficam de fora, porque o preço por kWh não faz sentido neles.
- A mediana mistura todas as classes. Para um corte mais fino, use o painel por classe da seção 5.

**Achado:**

| Período | Bandeira | % das faturas | Preço mediano (R$/kWh) |
|---|---|---|---|
| P0 | Verde | **100%** (536.267) | 0,5309 |
| P1 | Verde | 33,23% (177.282) | 0,5857 |
| P1 | Amarela | 25,14% (134.006) | 0,6045 |
| P1 | Vermelha P1 | 24,95% (133.097) | 0,6234 |
| P1 | Vermelha P2 | 16,68% (88.994) | 0,6577 |

- No primeiro ano **nenhuma fatura pagou adicional de bandeira**; nos últimos 12 meses, 67% pagaram.
- O preço mediano da **Verde** sobe de 0,5309 para 0,5857 (**+10,3%**), perto dos 10,5% de tarifa. O resto do aumento do preço médio (+12,45%) vem da bandeira e do mix de classes.
- A `silver.faturas` não traz o valor do adicional (só o nome da bandeira), então o efeito é **estimado** pelo preço por kWh. A mediana de todas as classes mistura os preços; um corte mais fino é o por classe da seção 5 (no residencial, +13,0% contra 10,5% de tarifa, ~2,3% de bandeira).

In [0]:
bandeiras = (
    fato.filter("periodo IS NOT NULL AND NOT troca_medidor AND consumo_kwh > 0")
    .withColumn("preco", F.col("valor_energia").cast("double") / F.col("consumo_kwh"))
    .groupBy("periodo", "bandeira_tarifaria")
    .agg(F.count("*").alias("faturas"),
         F.round(F.expr("percentile_approx(preco, 0.5)"), 4).alias("preco_kwh_mediano"))
    .orderBy("periodo", "bandeira_tarifaria")
)
display(bandeiras)

b = bandeiras.toPandas()
tot = b.groupby("periodo")["faturas"].transform("sum")
b["pct_faturas"] = (b["faturas"] / tot * 100).round(2)
print(b.pivot(index="bandeira_tarifaria", columns="periodo", values="pct_faturas").to_string())

periodo,bandeira_tarifaria,faturas,preco_kwh_mediano
P0,Verde,536267,0.5309
P1,Amarela,134096,0.6045
P1,Verde,177282,0.5857
P1,Vermelha P1,133097,0.6234
P1,Vermelha P2,88994,0.6577


periodo                P0     P1
bandeira_tarifaria              
Amarela               NaN  25.14
Verde               100.0  33.23
Vermelha P1           NaN  24.95
Vermelha P2           NaN  16.68


## 8. Geração distribuída (painel solar)

**O que explora:** o case diz que a energia gerada por painel solar é **abatida do consumo faturado**, o que pode explicar a queda de volume. Compara o consumo das UCs **com GD** e **sem GD** em três recortes:
- **8.1** ano contra ano, nos dez meses normais de P1 (sem novembro/2025 e fevereiro/2026);
- **8.2** a comparação do diretor, P1 contra P0, com todos os meses;
- **8.3** o **antes e depois** da conexão, UC por UC, que é o teste mais forte.

**Como ler:** se as UCs com GD caírem muito e as demais ficarem estáveis, a GD é um candidato forte. É uma **associação**: nenhum campo confirma a causa. A lista de UCs com GD vem da Silver (se existir) ou da Bronze (sem o cabeçalho repetido).

In [0]:
try:
    gd = spark.table(tabela_gd).select("id_uc").distinct().withColumn("tem_gd", F.lit(True))
    origem_gd = "Silver de GD"
    gd.count()
except Exception as e:
    print(f"Silver de GD indisponível ({str(e)[:90]}...). Usando a Bronze.")
    gd = (spark.table(f"{catalog}.bronze.geracao_distribuida")
            .filter(F.col("id_uc") != "id_uc")   # cabeçalho repetido
            .select("id_uc").distinct().withColumn("tem_gd", F.lit(True)))
    origem_gd = "Bronze"
print(f"origem: {origem_gd} | UCs com GD: {gd.count():,}  (esperado: ~3.510)")

origem: Silver de GD | UCs com GD: 3,510  (esperado: ~3.510)


### 8.1 Ano contra ano, nos meses normais

**Achado:** 3.510 UCs têm GD (3.127 residenciais e 383 comerciais), 7,8% das UCs. Consumo dos dez meses normais de P1 contra os mesmos meses do ano anterior:

| Classe | Com GD | Sem GD | kWh perdidos | % da queda |
|---|---|---|---|---|
| Residencial | **−40,5%** | −1,2% | com GD −2,12 mi; sem GD −0,65 mi | 39,9% e 12,3% |
| Comercial | **−36,9%** | −3,4% | com GD −1,38 mi; sem GD −1,20 mi | 26,1% e 22,6% |
| Industrial, rural e poder público | não têm GD | de −0,1% a +0,1% | ≈ 0 | ≈ 0 |

A queda total dos meses normais é de 5.312.292 kWh, e as UCs com GD respondem por **66,0%** dela (5% do consumo).

In [0]:
normais = [m for m in meses_p1 if m not in ATIPICOS]
normais_ant = [f"{int(m[:4]) - 1}-{m[5:]}" for m in normais]

d = (spark.table(f"{silver_schema}.leituras").join(uc, "id_uc", "left").join(gd, "id_uc", "left")
       .withColumn("tem_gd", F.coalesce("tem_gd", F.lit(False)))
       .withColumn("periodo", F.when(F.col("mes_referencia").isin(normais), "atual")
                               .when(F.col("mes_referencia").isin(normais_ant), "anterior"))
       .filter("periodo IS NOT NULL")
       .groupBy("classe", "tem_gd").pivot("periodo", ["anterior", "atual"])
       .agg(F.sum("consumo_kwh").alias("kwh"), F.countDistinct("id_uc").alias("ucs")))

(d.withColumn("variacao_pct", F.round((F.col("atual_kwh") / F.col("anterior_kwh") - 1) * 100, 1))
  .select("classe", "tem_gd", "anterior_ucs", "atual_ucs", "variacao_pct").orderBy("classe", "tem_gd").show())

dd = d.withColumn("delta_kwh", F.col("atual_kwh") - F.col("anterior_kwh"))
tot_n = dd.agg(F.sum("delta_kwh")).first()[0]
(dd.withColumn("pct_da_queda_total", F.round(F.col("delta_kwh") / F.lit(tot_n) * 100, 1))
   .select("classe", "tem_gd", "anterior_kwh", "atual_kwh", "delta_kwh", "pct_da_queda_total")
   .orderBy("delta_kwh").show(truncate=False))
print("variação total dos meses normais (kWh):", tot_n)

+-------------+------+------------+---------+------------+
|       classe|tem_gd|anterior_ucs|atual_ucs|variacao_pct|
+-------------+------+------------+---------+------------+
|    COMERCIAL| false|        4519|     4526|        -3.4|
|    COMERCIAL|  true|         383|      383|       -36.9|
|   INDUSTRIAL| false|         877|      877|         0.1|
|PODER PUBLICO| false|         222|      222|        -0.1|
|  RESIDENCIAL| false|       35156|    35209|        -1.2|
|  RESIDENCIAL|  true|        3126|     3127|       -40.5|
|        RURAL| false|         656|      656|        -0.1|
+-------------+------+------------+---------+------------+

+-------------+------+------------+---------+---------+------------------+
|classe       |tem_gd|anterior_kwh|atual_kwh|delta_kwh|pct_da_queda_total|
+-------------+------+------------+---------+---------+------------------+
|RESIDENCIAL  |true  |5240956     |3118836  |-2122120 |39.9              |
|COMERCIAL    |true  |3753225     |2368242  |-1384

### 8.2 Primeiro ano contra últimos 12 meses (a comparação do diretor)

**Achado:** com todos os meses, as UCs com GD perdem **mais do que a queda total**, e o resto da base cresce:

| Grupo | P0 | P1 | Variação |
|---|---|---|---|
| UCs **com** GD | 18,30 mi kWh | 7,10 mi kWh | **−61,2%** (−11,20 mi) |
| UCs **sem** GD | 208,75 mi kWh | 210,98 mi kWh | **+1,1%** (+2,22 mi) |
| **Total** | 227,05 mi kWh | 218,08 mi kWh | −3,95% (−8,97 mi) |

- Por classe, as UCs com GD caem **−61,4% no residencial** (−6,50 mi kWh) e **−60,9% no comercial** (−4,70 mi kWh). As sem GD: residencial +3,0%, comercial −1,3%, industrial +0,6%, rural +2,5%, poder público +2,4%.
- **Sem os painéis, o volume teria subido ~1%**, coerente com a base ter crescido ~0,6%.
- **Em reais** (preço médio do primeiro ano, R$ 0,519/kWh): o efeito-volume de −R$ 4,66 mi se divide em **−R$ 5,81 mi nas UCs com GD** e **+R$ 1,15 mi nas demais**. A soma bate com o efeito-volume da seção 4.
- A participação da GD no volume cai de 8,1% (P0) para 3,3% (P1).
- No P0, antes de qualquer painel, as UCs que depois instalaram consumiam bem mais que as demais: residencial ~282 contra ~169 kWh por mês, comercial ~1.678 contra ~814 (contas aproximadas). A adoção se concentra em clientes de alto consumo.

In [0]:
gd_f = (spark.table(f"{silver_schema}.leituras").join(uc, "id_uc", "left").join(gd, "id_uc", "left")
          .withColumn("tem_gd", F.coalesce("tem_gd", F.lit(False)))
          .withColumn("periodo", F.when(F.col("mes_referencia").between(*P0), "P0")
                                  .when(F.col("mes_referencia").between(*P1), "P1"))
          .filter("periodo IS NOT NULL"))

r = (gd_f.groupBy("classe", "tem_gd").pivot("periodo", ["P0", "P1"]).agg(F.sum("consumo_kwh"))
         .withColumn("delta_kwh", F.col("P1") - F.col("P0")))
tot = r.agg(F.sum("delta_kwh")).first()[0]

(r.withColumn("var_pct", F.round((F.col("P1") / F.col("P0") - 1) * 100, 1))
  .withColumn("pct_da_variacao_total", F.round(F.col("delta_kwh") / F.lit(tot) * 100, 1))
  .orderBy("delta_kwh").show(truncate=False))
print("variação total P1 vs P0 (kWh):", tot)

+-------------+------+--------+--------+---------+-------+---------------------+
|classe       |tem_gd|P0      |P1      |delta_kwh|var_pct|pct_da_variacao_total|
+-------------+------+--------+--------+---------+-------+---------------------+
|RESIDENCIAL  |true  |10584372|4087182 |-6497190 |-61.4  |72.4                 |
|COMERCIAL    |true  |7714129 |3013928 |-4700201 |-60.9  |52.4                 |
|COMERCIAL    |false |44120082|43557108|-562974  |-1.3   |6.3                  |
|RURAL        |false |2242780 |2299262 |56482    |2.5    |-0.6                 |
|PODER PUBLICO|false |4111425 |4211184 |99759    |2.4    |-1.1                 |
|INDUSTRIAL   |false |87177795|87690540|512745   |0.6    |-5.7                 |
|RESIDENCIAL  |false |71102344|73219850|2117506  |3.0    |-23.6                |
+-------------+------+--------+--------+---------+-------+---------------------+

variação total P1 vs P0 (kWh): -8973873


In [0]:
# Divisão do efeito-volume da seção 4 entre UCs com GD e demais (usa o preço médio de P0, Pm0)
rp = r.toPandas()
delta_gd = rp.loc[rp["tem_gd"], "delta_kwh"].sum()
delta_resto = rp.loc[~rp["tem_gd"], "delta_kwh"].sum()
print(f"UCs com GD:  {delta_gd:,.0f} kWh -> R$ {delta_gd * Pm0:,.0f}")
print(f"demais UCs:  {delta_resto:,.0f} kWh -> R$ {delta_resto * Pm0:,.0f}")
print(f"soma:        R$ {(delta_gd + delta_resto) * Pm0:,.0f}  (efeito-volume da seção 4: R$ {ef_volume:,.0f})")
print(f"UCs com GD: P0 {rp.loc[rp['tem_gd'], 'P0'].sum() / 1e6:.2f} mi kWh -> P1 {rp.loc[rp['tem_gd'], 'P1'].sum() / 1e6:.2f} mi kWh")

UCs com GD:  -11,197,391 kWh -> R$ -5,810,949
demais UCs:  2,223,518 kWh -> R$ 1,153,907
soma:        R$ -4,657,042  (efeito-volume da seção 4: R$ -4,657,042)
UCs com GD: P0 18.30 mi kWh -> P1 7.10 mi kWh


### 8.3 Antes e depois da conexão

**O que faz:** para cada UC com GD, alinha os meses pela data de conexão (mês 0 = mês da conexão) e calcula o consumo médio de todas as UCs em cada mês relativo. É o teste mais forte, porque compara a mesma UC antes e depois. A data de conexão vem da Bronze, em formato único `yyyy-MM-dd` (3.510 UCs, todas com data válida).

**Achado:** o consumo **cai abruptamente na conexão**.
- Nos 12 meses antes, o consumo médio fica entre 415 e 461 kWh por UC e mês (média ~438, com a onda da sazonalidade).
- No mês 0 (conexão no meio do mês) é 291,7, e **a partir do mês +1 cai para ~133 kWh** e fica entre 128 e 143 nos 12 meses seguintes (média ~135). **Queda de ~69%** (de ~438 para ~135), logo na conexão, sem tendência de volta.
- A quantidade de UCs em cada ponto vai de ~3.500 (meses antes) a 2.592 (mês +12), porque as conexões mais recentes ainda não têm 12 meses depois. A média mistura as classes residencial e comercial.
- Limites: a estação do ano mexe no consumo (as UCs conectam em meses diferentes, o que dilui, mas não elimina), e a comparação é descritiva. O padrão (estável antes, queda brusca na conexão) é compatível com o painel abatendo o consumo faturado, mas **não prova a causa**.

In [0]:
g = spark.table(f"{catalog}.bronze.geracao_distribuida").filter(F.col("id_uc") != "id_uc")
g.withColumn("padrao", F.regexp_replace("data_conexao", "[0-9]", "9")).groupBy("padrao").count().show()

conx = (g.select("id_uc", F.expr("try_to_date(data_conexao, 'yyyy-MM-dd')").alias("dt_con"))
          .dropDuplicates(["id_uc"]))
print("conexões sem data válida:", conx.filter("dt_con IS NULL").count())

ev = (spark.table(f"{silver_schema}.leituras")
        .join(conx.filter("dt_con IS NOT NULL"), "id_uc")
        .withColumn("mes_leit", F.expr("to_date(concat(mes_referencia, '-01'))"))
        .withColumn("meses_desde_conexao", F.floor(F.months_between("mes_leit", F.trunc("dt_con", "month"))))
        .filter(F.col("meses_desde_conexao").between(-12, 12))
        .groupBy("meses_desde_conexao")
        .agg(F.countDistinct("id_uc").alias("ucs"), F.round(F.avg("consumo_kwh"), 1).alias("consumo_medio"))
        .orderBy("meses_desde_conexao"))
ev.show(30)

evp = ev.toPandas().set_index("meses_desde_conexao")["consumo_medio"]
antes, depois = evp.loc[-12:-1].mean(), evp.loc[1:12].mean()
print(f"consumo médio antes (meses -12 a -1): {antes:.1f} | depois (meses +1 a +12): {depois:.1f} | variação {pct(depois / antes - 1)}")

+----------+-----+
|    padrao|count|
+----------+-----+
|9999-99-99| 3510|
+----------+-----+

conexões sem data válida: 0
+-------------------+----+-------------+
|meses_desde_conexao| ucs|consumo_medio|
+-------------------+----+-------------+
|                -12|3500|        446.3|
|                -11|3500|        437.4|
|                -10|3501|        427.6|
|                 -9|3501|        420.1|
|                 -8|3502|        415.3|
|                 -7|3502|        417.8|
|                 -6|3502|        424.7|
|                 -5|3502|        437.6|
|                 -4|3503|        448.2|
|                 -3|3504|        458.3|
|                 -2|3505|        460.5|
|                 -1|3505|        457.5|
|                  0|3505|        291.7|
|                  1|3456|        133.3|
|                  2|3404|        131.9|
|                  3|3352|        130.5|
|                  4|3290|        127.6|
|                  5|3222|        128.2|
|              

## 9. Resumo para a daily e para o diagnóstico

| Pergunta | Resposta |
|---|---|
| O volume da Silver reproduz o "quase −4%"? | **Sim: −3,95%** (opção A, `consumo = leitura_atual` nas trocas de medidor). |
| A receita reproduz o "+8%"? | **Sim: +8,00%**, com `valor_energia` ou `valor_total`. |
| O tratamento das trocas de medidor muda a conclusão? | Pouco por UC (A −4,58%, B −4,62%). No total, só A reproduz o −4%; o consumo original (C) dá −174%. |
| Quanto da variação da receita é preço e quanto é volume? | Receita +R$ 9,43 mi = **efeito-preço +R$ 14,09 mi** + **efeito-volume −R$ 4,66 mi**. |
| O preço médio subiu mais do que os 10,5% da tarifa residencial? | **Sim: +12,45%.** A Verde subiu ~10,3% (tarifa); o resto vem da bandeira (no primeiro ano 100% Verde; nos últimos 12 meses, 67% pagam adicional) e do mix. No residencial, +13,0% (~2,3% de bandeira, estimado). |
| Onde o volume mais caiu? | **Comercial (−10,2%) e residencial (−5,4%)**. Industrial, rural e poder público ficam estáveis. |
| A queda existe nas UCs estáveis, sem troca de medidor? | Sim: −4,58% por UC-mês (38.397 UCs). |
| A GD explica parte da queda? | **Sim, a maior parte:** as 3.510 UCs com painel caem **−61,2%** (−11,2 mi kWh, **−R$ 5,81 mi** em receita de energia), e as demais crescem +1,1%. Antes e depois da conexão, o consumo cai ~69% logo no mês seguinte. |
| Há meses fora da tendência? | **Novembro/2025 e fevereiro/2026**: consumo +8% sobre o ano anterior em todas as classes (outros meses: −1% a −4%). Se seguissem a tendência, a queda do P1 seria de ~−6%. A causa não está nos dados. |

**Hipóteses, não conclusões:** a análise localiza onde o volume cai e mede o tamanho de cada efeito, mas nenhum campo prova a causa. O elo entre a GD e a queda é uma associação forte e coerente com o que o case descreve (a geração é abatida do consumo faturado). A causa dos meses atípicos (calor? outra?) e a de 2025-07 na rede (evento climático em três alimentadores) ficam para a V-Energy confirmar.

**Para o contrato da Gold:**
- `valor_energia` e `valor_total` dão a mesma variação de receita, então a escolha de `valor_energia` (sem tributos) se sustenta.
- O `consumo_kwh` dos meses de troca de medidor é a `leitura_atual`, regra que reproduz o −4%; o dicionário deve dizer isso.
- A `receita_decomposicao` precisa de **corte por classe** e de uma marca de **GD por mês** (antes e depois da conexão), e o efeito-preço pode ser separado em tarifa e bandeira (a Verde como referência).
- Os indicadores mensais precisam do corte por classe e município, para mostrar, por exemplo, os meses atípicos e a queda residencial.

**O que ficou de fora:** município e bairro (dependem da Silver de UCs), a pergunta "qual bairro mais perdeu volume residencial?" e a causa dos meses atípicos.